<a href="https://colab.research.google.com/github/vinkoff/Learner/blob/master/Finance_Assignment.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# AI Prototype - Finance Earnings Summarization
### Model Selection, Prompt Engineering, Evaluation Metrics, Score Improvement Lab


## Step 1 - Install and Import


In [ ]:
!pip install anthropic rouge-score nltk --quiet

import os, time, math, anthropic, nltk, warnings
import matplotlib.pyplot as plt
import numpy as np
from collections import defaultdict
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from rouge_score import rouge_scorer
from nltk.tokenize import word_tokenize
warnings.filterwarnings("ignore")
nltk.download("punkt", quiet=True)
nltk.download("punkt_tab", quiet=True)
print("Libraries ready")


## Step 2 - API Setup


In [ ]:
# from google.colab import userdata
# api_key = userdata.get("ANTHROPIC_API_KEY")
api_key = os.environ.get("ANTHROPIC_API_KEY", "")
client = anthropic.Anthropic(api_key=api_key)
MODEL_FAST = "claude-haiku-4-5"
MODEL_POWERFUL = "claude-sonnet-4-6"
print(MODEL_FAST, MODEL_POWERFUL)


## Step 3 - Finance Earnings Call Dataset


In [ ]:
incidents = [
    {"id": 1, "type": "Revenue Miss", "text": "Apex Cloud reported Q3 revenue of $4.2 billion versus analyst expectations of $4.6 billion. The miss was caused by a service outage that increased customer churn in the cloud segment. Operating margin fell from 32 percent to 26 percent. Management announced a $500 million share buyback and guided next quarter below Wall Street estimates.", "reference_summary": "Apex Cloud missed Q3 revenue expectations at $4.2 billion versus $4.6 billion due to outage related cloud churn, margin fell to 26 percent, and management announced a $500 million buyback with weak next quarter guidance.", "alt_references": ["Apex Cloud posted a Q3 revenue miss of $4.2 billion against a $4.6 billion estimate because of cloud churn after a service outage, while margins dropped and a $500 million buyback was announced.", "Q3 revenue came in below expectations for Apex Cloud as outage driven churn hurt results, margins contracted to 26 percent, and next quarter guidance was set below consensus with a new buyback plan."]},
    {"id": 2, "type": "Guidance Upgrade", "text": "MedCore reported Q2 revenue of $2.8 billion, above consensus of $2.5 billion, and adjusted EPS of $3.42 above the expected $3.10. Growth was driven by FDA approval of its oncology drug Velixatinib. The company raised full year revenue guidance from $9.8 billion to $10.6 billion and raised EPS guidance from $11.20 to $12.40.", "reference_summary": "MedCore beat Q2 estimates with $2.8 billion in revenue and $3.42 EPS on Velixatinib strength, then raised full year revenue guidance to $10.6 billion and EPS guidance to $12.40.", "alt_references": ["MedCore delivered a Q2 beat driven by Velixatinib approval and raised both revenue and EPS guidance for the full year.", "Strong Q2 results at MedCore topped expectations and led management to increase full year outlook for both revenue and earnings."]},
    {"id": 3, "type": "Margin Expansion", "text": "BrightMart reported Q1 gross margin of 38.4 percent, up 420 basis points from last year and the best level in five years. Management said AI based inventory optimization reduced shrink and lower freight costs helped profitability. Digital sales rose 67 percent to $1.1 billion and operating income increased 44 percent to $890 million. The company plans 180 net new store openings this year.", "reference_summary": "BrightMart expanded Q1 gross margin to 38.4 percent through AI inventory gains and lower freight costs, while digital sales jumped to $1.1 billion, operating income rose 44 percent, and management kept aggressive store expansion plans.", "alt_references": ["BrightMart posted strong Q1 margin expansion helped by AI inventory optimization, with fast digital growth and higher operating income supporting store expansion plans.", "Q1 results from BrightMart showed five year high gross margin, strong digital sales growth, and rising operating profit driven by inventory and freight improvements."]},
    {"id": 4, "type": "Merger Announcement", "text": "TechGiant announced an all stock acquisition of NimbusTech for $7.8 billion at a 42 percent premium. Management said the acquisition would add $600 million in annual recurring revenue and accelerate the company AI roadmap by 18 to 24 months. TechGiant also reported Q4 revenue in line with estimates and EPS of $2.87, which beat consensus by $0.12. Full year revenue guidance was set at $74 billion to $76 billion including the deal contribution.", "reference_summary": "TechGiant announced a $7.8 billion all stock acquisition of NimbusTech that adds $600 million in recurring revenue and strengthens its AI strategy, while Q4 EPS beat estimates and full year guidance included deal impact.", "alt_references": ["A $7.8 billion NimbusTech acquisition was announced by TechGiant alongside an EPS beat and full year guidance that included expected merger contribution.", "TechGiant used its earnings call to reveal a large all stock deal for NimbusTech, adding recurring revenue and AI roadmap benefits on top of a Q4 EPS beat."]}
]
print(f"{len(incidents)} finance events loaded")
for i in incidents:
    print(i["id"], i["type"])


## Step 4 - Prompt Engineering


In [ ]:
def make_prompt(strategy, text):
    if strategy == "basic":
        return f"Summarize this earnings call event in one sentence:\n\n{text}"
    elif strategy == "structured":
        return f"""You are a financial analyst writing a short earnings summary.
Summarize in 1 to 2 sentences. Always include:
- Result versus expectations
- Main business driver
- Forward guidance or outlook
- Any corporate action if present

Event: {text}
Summary:"""
    elif strategy == "few_shot":
        return f"""Write a concise financial analyst summary.

Example:
Event: SoftTech reported revenue above estimates, raised guidance, and announced a buyback.
Summary: SoftTech beat expectations, raised guidance, and announced a buyback.

Now summarize:
Event: {text}
Summary:"""
print("Prompt functions ready")


## Step 5 - Generate Baseline Summaries


In [ ]:
def call_model(model, prompt, temperature=1.0):
    params = dict(model=model, max_tokens=250, messages=[{"role": "user", "content": prompt}])
    if temperature != 1.0:
        params["temperature"] = temperature
    start = time.time()
    response = client.messages.create(**params)
    return {"summary": response.content[0].text.strip(), "latency_s": round(time.time() - start, 2), "tokens": response.usage.input_tokens + response.usage.output_tokens}

strategies = ["basic", "structured", "few_shot"]
models = {"haiku": MODEL_FAST, "sonnet": MODEL_POWERFUL}
results = []

for incident in incidents:
    for strategy in strategies:
        prompt = make_prompt(strategy, incident["text"])
        for model_label, model_id in models.items():
            out = call_model(model_id, prompt)
            results.append({
                "incident_id": incident["id"],
                "type": incident["type"],
                "strategy": strategy,
                "model": model_label,
                "summary": out["summary"],
                "reference": incident["reference_summary"],
                "alt_references": incident["alt_references"],
                "latency_s": out["latency_s"],
                "tokens": out["tokens"]
            })
print(f"Generated {len(results)} summaries")


## Step 6 - Evaluation Functions


In [ ]:
def compute_bleu(refs, hyp, multi=False):
    hyp_tok = word_tokenize(hyp.lower())
    if multi:
        ref_tok = [word_tokenize(r.lower()) for r in refs]
    else:
        ref_tok = [word_tokenize(refs.lower())]
    return round(sentence_bleu(ref_tok, hyp_tok, weights=(1,0,0,0), smoothing_function=SmoothingFunction().method1), 4)

def compute_rouge(ref, hyp):
    s = rouge_scorer.RougeScorer(["rouge1", "rougeL"], use_stemmer=True).score(ref, hyp)
    return {"rouge1": round(s["rouge1"].fmeasure, 4), "rougeL": round(s["rougeL"].fmeasure, 4)}

def compute_rouge_best(refs, hyp):
    scores = [compute_rouge(r, hyp)["rouge1"] for r in refs]
    return round(max(scores), 4)

def perplexity_proxy(text):
    ngrams = [text[i:i+3] for i in range(len(text)-2)]
    from collections import Counter
    counts = Counter(ngrams)
    total = sum(counts.values())
    probs = [c/total for c in counts.values()]
    entropy = -sum(p * math.log2(p) for p in probs if p > 0)
    return round(2**entropy, 2)

def score_result(r, multi_ref=False):
    refs = [r["reference"]] + r["alt_references"]
    r["bleu"] = compute_bleu(refs if multi_ref else r["reference"], r["summary"], multi=multi_ref)
    rouge = compute_rouge(r["reference"], r["summary"])
    r["rouge1"] = rouge["rouge1"]
    r["rougeL"] = rouge["rougeL"]
    r["rouge1_best"] = compute_rouge_best(refs, r["summary"])
    r["perplexity"] = perplexity_proxy(r["summary"])
    return r

for r in results:
    score_result(r)
print("Metrics computed")


## Step 7 - Baseline Results Table


In [ ]:
def print_table(data):
    grouped = defaultdict(lambda: {"bleu": [], "rouge1": [], "rougeL": [], "perplexity": [], "latency_s": []})
    for r in data:
        key = (r["model"], r["strategy"])
        for m in ["bleu", "rouge1", "rougeL", "perplexity", "latency_s"]:
            grouped[key][m].append(r[m])
    hdr = "{:<10} {:<12} {:>6} {:>8} {:>8} {:>8} {:>7}".format("Model", "Strategy", "BLEU", "ROUGE-1", "ROUGE-L", "Perplx", "Lat(s)")
    print(hdr)
    print("-" * len(hdr))
    for (model, strategy), vals in sorted(grouped.items()):
        avg = lambda k: round(sum(vals[k]) / len(vals[k]), 3)
        line = "{:<10} {:<12} {:>6.3f} {:>8.3f} {:>8.3f} {:>8.1f} {:>7.2f}".format(model, strategy, avg("bleu"), avg("rouge1"), avg("rougeL"), avg("perplexity"), avg("latency_s"))
        print(line)
    return grouped

baseline_grouped = print_table(results)


## Step 8 - Baseline Visualizations


In [ ]:
strategies_list = ["basic", "structured", "few_shot"]
x = np.arange(len(strategies_list))
width = 0.35
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
fig.suptitle("Finance Domain Baseline Scores", fontsize=13)
metrics = ["bleu", "rouge1", "rougeL"]
for idx, metric in enumerate(metrics):
    ax = axes[idx]
    for m_idx, model_label in enumerate(["haiku", "sonnet"]):
        vals = []
        for strat in strategies_list:
            matching = [r[metric] for r in results if r["model"] == model_label and r["strategy"] == strat]
            vals.append(round(sum(matching) / len(matching), 3))
        offset = (m_idx - 0.5) * width
        ax.bar(x + offset, vals, width, label=model_label)
    ax.set_title(metric.upper())
    ax.set_xticks(x)
    ax.set_xticklabels(["Basic", "Structured", "Few-Shot"])
    ax.set_xlabel("Prompt Strategy")
    ax.set_ylabel("Score")
    ax.legend()
    ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()


## Step 9 - Knob 1: Temperature


In [ ]:
TEMP = 0.0
temp_results = []
for incident in incidents:
    out = call_model(MODEL_FAST, make_prompt("few_shot", incident["text"]), temperature=TEMP)
    row = {"incident_id": incident["id"], "type": incident["type"], "strategy": "few_shot", "model": f"haiku_t{TEMP}", "summary": out["summary"], "reference": incident["reference_summary"], "alt_references": incident["alt_references"], "latency_s": out["latency_s"], "tokens": out["tokens"]}
    score_result(row)
    temp_results.append(row)
print("Temperature test complete")


## Step 10 - Knob 2: Chain-of-Thought Prompting


In [ ]:
def make_cot_prompt(text):
    return f"""You are a financial analyst.
Step 1: Identify the result versus expectations.
Step 2: Identify the main driver.
Step 3: Identify guidance or outlook.
Step 4: Identify any corporate action.
Step 5: Write a short summary.

Event: {text}
Final summary:"""

cot_results = []
for incident in incidents:
    out = call_model(MODEL_FAST, make_cot_prompt(incident["text"]), temperature=0.0)
    row = {"incident_id": incident["id"], "type": incident["type"], "strategy": "chain_of_thought", "model": "haiku_cot", "summary": out["summary"], "reference": incident["reference_summary"], "alt_references": incident["alt_references"], "latency_s": out["latency_s"], "tokens": out["tokens"]}
    score_result(row)
    cot_results.append(row)
print("CoT test complete")


## Step 11 - Knob 3: Multi-Reference Scoring


In [ ]:
for r in results:
    if r["strategy"] == "few_shot" and r["model"] == "haiku":
        refs = [r["reference"]] + r["alt_references"]
        print(r["incident_id"], r["type"], r["rouge1"], compute_rouge_best(refs, r["summary"]))


## Step 12 - Knob 4: Extended Few-Shot


In [ ]:
def make_extended_few_shot(text):
    return f"""Write a concise financial analyst summary.

Example 1:
Event: SoftTech beat revenue estimates, raised guidance, and announced a buyback.
Summary: SoftTech beat expectations, raised guidance, and announced a buyback.

Example 2:
Event: DeltaForge missed revenue expectations because of a factory strike and suspended guidance.
Summary: DeltaForge missed expectations due to production disruption and suspended guidance.

Example 3:
Event: ViewMax beat subscriber estimates and raised outlook for next quarter.
Summary: ViewMax delivered a strong quarter and raised subscriber outlook.

Now summarize:
Event: {text}
Summary:"""

extended_fs_results = []
for incident in incidents:
    out = call_model(MODEL_FAST, make_extended_few_shot(incident["text"]), temperature=0.0)
    row = {"incident_id": incident["id"], "type": incident["type"], "strategy": "extended_few_shot", "model": "haiku_ext", "summary": out["summary"], "reference": incident["reference_summary"], "alt_references": incident["alt_references"], "latency_s": out["latency_s"], "tokens": out["tokens"]}
    score_result(row)
    extended_fs_results.append(row)
print("Extended few-shot test complete")


## Step 13 - Full Improvement Comparison


In [ ]:
avg = lambda lst, k: round(sum(r[k] for r in lst) / len(lst), 3)
configurations = [("Baseline Basic", [r for r in results if r["strategy"] == "basic" and r["model"] == "haiku"]), ("Baseline Structured", [r for r in results if r["strategy"] == "structured" and r["model"] == "haiku"]), ("Baseline Few-Shot", [r for r in results if r["strategy"] == "few_shot" and r["model"] == "haiku"]), ("Temp Zero", temp_results), ("Chain-of-Thought", cot_results), ("Extended Few-Shot", extended_fs_results)]
for name, data in configurations:
    print(name, avg(data, "bleu"), avg(data, "rouge1"), avg(data, "rougeL"), avg(data, "perplexity"))


## Step 14 - Visualize the Improvement Journey


In [ ]:
labels = ["Basic", "Structured", "Few-Shot", "Temp 0", "CoT", "Extended"]
sets = [[r for r in results if r["strategy"] == "basic" and r["model"] == "haiku"], [r for r in results if r["strategy"] == "structured" and r["model"] == "haiku"], [r for r in results if r["strategy"] == "few_shot" and r["model"] == "haiku"], temp_results, cot_results, extended_fs_results]
avg = lambda lst, k: round(sum(r[k] for r in lst) / len(lst), 3)
rouge1_vals = [avg(d, "rouge1") for d in sets]
bleu_vals = [avg(d, "bleu") for d in sets]
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].bar(labels, rouge1_vals, color="steelblue")
axes[0].set_title("Finance Domain ROUGE-1 by Configuration")
axes[0].set_xlabel("Configuration")
axes[0].set_ylabel("ROUGE-1 Score")
axes[0].tick_params(axis="x", rotation=30)
axes[1].plot(labels, rouge1_vals, marker="o", label="ROUGE-1")
axes[1].plot(labels, bleu_vals, marker="s", label="BLEU")
axes[1].set_title("Finance Domain Score Improvement Journey")
axes[1].set_xlabel("Configuration")
axes[1].set_ylabel("Score")
axes[1].tick_params(axis="x", rotation=30)
axes[1].legend()
plt.tight_layout()
plt.show()


## Step 15 - Discussion Answers

**1. Which technique gave the biggest ROUGE-1 gain? Was it what you expected?**

Extended few-shot prompting gave the biggest improvement. That was mostly expected because more examples usually make the output format more consistent.

**2. Did Chain-of-Thought hurt or help perplexity? Why might reasoning steps affect fluency?**

Chain-of-Thought can hurt fluency because the model focuses on reasoning through the steps before giving the final answer. This can make the final summary less smooth or more repetitive.

**3. Why did multi-reference scoring improve numbers without changing the model at all?**

The model output stayed the same, but the scoring became more fair. There were multiple correct reference summaries, so the model had more chances to match valid wording.

**4. If you had to deploy one configuration for a real finance alert system, which would you pick and why?**

I would choose extended few-shot with temperature zero. It gives stable output and keeps the summaries consistent across similar financial events.

**5. What is the ceiling of prompt engineering? What would push you to fine-tune instead?**

Prompt engineering reaches a limit when the task needs special terminology, repeated structure, or internal company style that the model does not follow reliably. Fine-tuning would make sense if prompt changes no longer improve quality enough.
